# Width x Depth
---
## Part 1 — Width as Nonlinear Basis Expansion

### 1. Model Definition

Consider a 1-hidden-layer neural network:
$$
f(x) = W_2 \, \sigma(W_1 x + b_1) + b_2
$$
where:
- $x \in \mathbb{R}^d$ is the input vector  
- $W_1 \in \mathbb{R}^{m \times d}$  
- $m$ is the width (number of neurons in the hidden layer)  
- $\sigma(\cdot)$ is a nonlinear activation function (ReLU, sigmoid, etc.)

Each hidden neuron computes:
$$
a_i(x) = \sigma(w_i^T x + b_i)
$$
and the hidden representation is:
$$
a(x) =
\begin{bmatrix}
a_1(x) \\
\vdots \\
a_m(x)
\end{bmatrix}
\in \mathbb{R}^m
$$

For a scalar output, the network can be written as:
$$
f(x) = \sum_{i=1}^{m} \alpha_i \, \sigma(w_i^T x + b_i) + b_2
$$
where $\alpha_i$ are the output layer weights (components of $W_2$).

---

### 2. Linear Algebra Interpretation

Define the learned nonlinear feature map:
$$
\Phi(x) =
\begin{bmatrix}
\phi_1(x) \\
\vdots \\
\phi_m(x)
\end{bmatrix}
=
\begin{bmatrix}
\sigma(w_1^T x + b_1) \\
\vdots \\
\sigma(w_m^T x + b_m)
\end{bmatrix}
\in \mathbb{R}^m
$$

Then the network becomes:
$$
f(x) = W_2 \, \Phi(x) + b_2
$$

Thus, a wide shallow neural network is equivalent to a **linear model applied to a learned nonlinear feature space**.

---

### 3. Width as Learned Nonlinear Basis Expansion

The model can be expressed as:
$$
f(x) = \sum_{i=1}^{m} \alpha_i \, \phi_i(x),
\quad \text{with} \quad
\phi_i(x) = \sigma(w_i^T x + b_i)
$$

This has the same structure as classical basis expansions:
- Polynomial basis: $x^k$
- Fourier basis: $\sin(kx)$
- Neural networks: learned nonlinear basis functions $\phi_i(x)$

Increasing the width $m$ increases the number of learned basis functions, which enlarges the function class that the network can represent.

---

### 4. Geometric Interpretation

Each neuron defines a hyperplane in the input space:
$$
w_i^T x + b_i = 0
$$

After applying the activation function $\sigma$, this hyperplane induces a nonlinear transformation (or “bend”) in the input space.

Consequences of increasing width:
- More hyperplanes in the input space  
- More nonlinear regions  
- Finer partition of the domain  
- Higher approximation resolution of complex functions  

However, all transformations are still applied directly to the original input $x$.

---

### 5. Structural Limitation of Shallow Width

In a 1-hidden-layer network:
$$
a_i(x) = \sigma(w_i^T x + b_i)
$$
all hidden neurons operate directly on the raw input $x$, not on intermediate learned representations.

Therefore:
- No hierarchical feature construction  
- No feature reuse across layers  
- Only a single nonlinear transformation stage  
- Final output is a linear combination of nonlinear features  

Even as $m \to \infty$, the structure remains:
$$
f(x) = \sum_{i=1}^{m} \alpha_i \, \phi_i(x)
$$
which is expressive but non-hierarchical.

---

### 6. Key Takeaway (Part 1)

Width increases expressivity by expanding the dimensionality of a learned nonlinear feature map $\Phi(x)$, allowing the network to approximate complex functions through a richer set of nonlinear basis functions, while still operating as a shallow (non-compositional) model over the original input space.

---
### Experiment: Width and Expressivity

enhance writing:

To better illustrate the theoretical results from Part 1, we consider the approximation of a nonlinear target function defined on $\mathbb{R}^2$:

$$
f(x_1, x_2) = \sin(2x_1)\cos(2x_2)
$$

This function presents several properties that make it a suitable benchmark for studying width:
- Multiple local oscillations
- Nonlinear interactions between input dimensions
- Rapid variation across the input space

In this experiment, we keep the architecture shallow (one hidden layer) and vary only the width of the network. This isolates the role of width in expressivity and allows us to empirically test the hypothesis that:
$$
\text{larger width} \;\Rightarrow\; \text{richer nonlinear basis expansion} \;\Rightarrow\; \text{better approximation of complex functions}.
$$


In [7]:
import math
import torch
import numpy as np
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(42)
rng = np.random.default_rng(42)

In [ ]:
grid_size = 100
x1 = np.linspace(-2, 2, grid_size)
x2 = np.linspace(-2, 2, grid_size)

X1, X2 = np.meshgrid(x1, x2)
x_np = np.column_stack([X1.ravel(), X2.ravel()])
y_np = (np.sin(2 * x_np[:, 0]) * np.cos(2 * x_np[:, 1])).reshape(-1, 1)

print("X shape:", x_np.shape)
print("y shape:", y_np.shape)

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim=2, width=50):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(input_dim, width),
            nn.ReLU(),
            nn.Linear(width, 1)
        )         

    def forward(self, x):
        return self.model(x)

    def train(self, X, y, epochs=100, learning_rate=0.01):
        optimizer = torch.optim.Adam(self.model.parameters(), lr=learning_rate)
        loss_fn = nn.MSELoss()

        for epoch in range(epochs):
            optimizer.zero_grad()

            y_pred = model(X)
            loss = loss_fn(y_pred, y)

            loss.backward()
            optimizer.step()

            if epoch % 50 == 0:
                print(f"Epoch {epoch} | Loss: {loss.item():.6f}")  

model = MLP(width=5)
x = torch.from_numpy(x_np).float()
y = torch.from_numpy(y_np).float()
model.train(x, y, epochs=100)


In [ ]:
import matplotlib.pyplot as plt

with torch.no_grad():
    y_pred = model(X).numpy().reshape(grid_size, grid_size)

y_true = y_np.reshape(grid_size, grid_size)
x1_graph = x_np[:, 0].reshape(grid_size, grid_size)
x2_graph = x_np[:, 1].reshape(grid_size, grid_size)

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

ax.plot_surface(x1_graph, x2_graph, y_true, alpha=0.4)
ax.plot_surface(x1_graph, x2_graph, y_pred, alpha=0.4)
ax.set_box_aspect(aspect=None, zoom=0.9)

ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.set_zlabel("y")
ax.set_title("Surface Function Approximation Example")

In [ ]:
model = MLP(width=100)
model.train(x, y, epochs=100)

with torch.no_grad():
    y_pred = model(X).numpy().reshape(grid_size, grid_size)

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

ax.plot_surface(x1_graph, x2_graph, y_true, alpha=0.4)
ax.plot_surface(x1_graph, x2_graph, y_pred, alpha=0.4)
ax.set_box_aspect(aspect=None, zoom=0.9)

ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.set_zlabel("y")
ax.set_title("Surface Function Approximation Example")

## Part 2 — Width vs Parameter Efficiency

For a 1-hidden-layer network with width $m$:
$$
f(x) = \sum_{i=1}^{m} \alpha_i \, \sigma(w_i^T x + b_i),
$$
increasing width corresponds to increasing the number of learned nonlinear basis functions.

The parameter count for a shallow wide network with input dimension $d$ and width $m$ scales as:
$$
\#\theta \approx d m + m + m + 1 \sim O(dm),
$$
so enlarging width directly increases the number of parameters and the dimensionality of the learned feature map $\Phi(x) \in \mathbb{R}^m$.

From a parameter efficiency perspective, a wide shallow model relies on a flat basis expansion over the input space, where each neuron independently contributes to the approximation. There is no feature reuse or hierarchical transformation.

For compositional target functions of the form:
$$
f(x) = h(g(x)),
$$
a deep network can represent the structure through sequential transformations and reuse intermediate features across layers. In contrast, a shallow network must approximate the same composition using a single sum of basis functions, which may require a much larger width (and thus more parameters) to achieve similar accuracy.

Therefore, increasing width improves expressivity but is not always parameter-efficient, especially for functions with hierarchical or compositional structure, where depth can achieve comparable representations with fewer effective parameters through feature reuse.]

---

### Example: Approximating a Multi-Scale Nonlinear Function

To better evaluate the impact of width on expressivity and parameter efficiency, we consider the approximation of a multi-scale nonlinear function defined on $\mathbb{R}$:
$$
f(x) = \sin(5\sin(3x))
$$

Unlike simple smooth functions, this target combines multiple frequency components, introducing both low- and high-frequency structure. As a result, it requires a richer set of nonlinear basis functions to accurately capture fine oscillatory patterns across the input domain.

From the shallow network perspective,
$$
\hat{f}(x) = \sum_{i=1}^{m} \alpha_i \, \sigma(w_i x + b_i),
$$
the model approximates the function as a linear combination of learned nonlinear basis functions. Increasing the width $m$ increases the number of such basis functions, allowing the network to represent higher-frequency details more effectively.

In this experiment, we keep the architecture shallow and vary only the width of the hidden layer. This isolates the role of width and allows us to observe how limited width leads to underfitting of high-frequency components, while larger width improves the approximation by expanding the learned nonlinear feature space.

In [ ]:
grid_size = 10000
x_np = np.linspace(-np.pi, np.pi, grid_size).reshape(-1, 1)
y_np = (np.sin(5 * np.sin(3 * x_np))).reshape(-1, 1)

print("X shape:", x_np.shape)
print("y shape:", y_np.shape)

In [ ]:
x = torch.from_numpy(x_np).float()
y = torch.from_numpy(y_np).float()

model = MLP(input_dim=1, width=500)
model.train(x, y, epochs=1000)

In [ ]:
with torch.no_grad():
    y_pred = model(x).numpy()

fig, ax = plt.subplots(figsize=(6, 4))
plt.plot(x, y, label='True')
plt.plot(x, y_pred, label='Pred', color='red')

ax.set_xlabel('x')
ax.set_ylabel('y')

plt.legend()
plt.title('Approximation with 1 layer MLP (100 neurons)', fontweight='bold')
plt.show()

In [ ]:
class two_layers_MLP(nn.Module):
    def __init__(self, input_dim=1, width=50):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(input_dim, width),
            nn.ReLU(),
            nn.Linear(width, width),
            nn.ReLU(),
            nn.Linear(width, 1)
        )         

    def forward(self, x):
        return self.model(x)

    def train(self, X, y, epochs=100, learning_rate=0.01):
        optimizer = torch.optim.Adam(self.model.parameters(), lr=learning_rate)
        loss_fn = nn.MSELoss()

        for epoch in range(epochs):
            optimizer.zero_grad()

            y_pred = model(X)
            loss = loss_fn(y_pred, y)

            loss.backward()
            optimizer.step()

            if epoch % 50 == 0:
                print(f"Epoch {epoch} | Loss: {loss.item():.6f}") 

model = two_layers_MLP()
model.train(x, y, epochs=1000)

In [ ]:
with torch.no_grad():
    y_pred = model(x).numpy()

fig, ax = plt.subplots(figsize=(6, 4))
plt.plot(x, y, label='True')
plt.plot(x, y_pred, label='Pred', color='red')

ax.set_xlabel('x')
ax.set_ylabel('y')

plt.legend()
plt.title('Approximation with 1 layer MLP (100 neurons)', fontweight='bold')
plt.show()

## Part 3 - Neural Tangent Kernels (NTKs) and the Infinite-Width Regime

A wide neural network can be viewed as a nonlinear basis expansion:

$$
f(x, \theta) = \sum_{i=1}^{m} \alpha_i \, \phi_i(x; \theta),
$$

where increasing the width $m$ increases the number of nonlinear features available to represent functions. As $m$ grows, the network can approximate increasingly complex functions by combining many nonlinear units.

However, to understand the infinite-width regime, it is more insightful to shift perspective: instead of viewing the network as nonlinear in $x$, we analyze it as a function of its **parameters**,

$$
f(x, \theta), \quad \theta \in \mathbb{R}^{P}.
$$

During training, the dataset $\{x_i\}_{i=1}^n$ is fixed and only the parameters $\theta$ evolve. The learning dynamics therefore take place in **parameter space**.

---

### 1. Infinite Width as a Rich Random Basis

When $m$ is very large, the network at initialization already contains a vast collection of random nonlinear features. In this regime, the randomly initialized functions span a rich portion of the function space.

Intuitively:

- A narrow network must *learn representations*.
- A very wide network already contains many useful nonlinear features at initialization.
- Training mainly adjusts the **linear combination** of these features.

In other words, when $m \to \infty$, the network behaves less like a model that reshapes its internal geometry and more like a model that reweights an already rich basis.

---

### 2. Linearization in Parameter Space

To formalize this idea, we perform a first-order Taylor expansion of the network around its initialization $\theta_0$:

$$
f(x, \theta) 
\approx 
f(x, \theta_0) 
+ 
\nabla_\theta f(x, \theta_0)^T (\theta - \theta_0).
$$

This expansion is taken **with respect to the parameters**, not the inputs.

Define the gradient feature map:

$$
\psi(x) := \nabla_\theta f(x, \theta_0) \in \mathbb{R}^{P}.
$$

Then the linearized model becomes:

$$
f(x, \theta)
\approx
f(x, \theta_0) + \psi(x)^T (\theta - \theta_0).
$$

---

### 3. Why Is the Taylor Expansion Valid?

For a smooth parameterized model, the Taylor expansion is locally valid whenever $\theta$ remains close to $\theta_0$.

The key observation in the NTK regime is that as $m \to \infty$:

1. Parameter updates during gradient descent scale like $O(1/\sqrt{m})$.
2. Individual parameters move very little.
3. The gradient feature map changes negligibly:

$$
\nabla_\theta f(x, \theta_t)
\approx
\nabla_\theta f(x, \theta_0).
$$

Thus, the training trajectory remains in a small neighborhood of initialization in parameter space, making the first-order approximation accurate throughout training.

This is sometimes called the **lazy training regime**: the network does not significantly change its internal representations — it stays close to its tangent space.

---

### 4. Linear Regression Interpretation

Let

$$
\Delta \theta = \theta - \theta_0.
$$

Then the model becomes

$$
f(x) 
\approx 
f(x, \theta_0) + \psi(x)^T \Delta \theta.
$$

This has the exact form of a linear regression model:

$$
\hat{y}(x) = w^T \phi(x) + b,
$$

with the correspondence:

$$
\phi(x) \leftrightarrow \psi(x) = \nabla_\theta f(x, \theta_0),
$$

$$
w \leftrightarrow \Delta \theta,
$$

$$
b \leftrightarrow f(x, \theta_0).
$$

Therefore, training the wide neural network in the NTK regime is equivalent to learning a **linear model in a high-dimensional feature space**, where the features are gradient features evaluated at initialization.

Importantly:

- The features $\psi(x)$ are fixed.
- Only their linear combination $\Delta \theta$ is learned.

This formalizes the earlier intuition: infinite-width networks already map inputs into a rich feature space; training only reweights those features.

---

### 5. From Linear Regression to a Kernel

In linear regression with feature map $\psi(x)$, predictions for a dataset $\{x_i\}_{i=1}^n$ depend only on inner products:

$$
\psi(x_i)^T \psi(x_j).
$$

This induces a kernel:

$$
K(x, x')
=
\psi(x)^T \psi(x')
=
\nabla_\theta f(x, \theta_0)^T
\nabla_\theta f(x', \theta_0).
$$

This is the **Neural Tangent Kernel (NTK)**.

As width tends to infinity:

$$
K(x, x'; \theta_t)
\approx
K(x, x'; \theta_0),
$$

meaning the kernel remains effectively constant during training.

Thus, gradient descent on a wide neural network becomes equivalent to **kernel regression** with kernel $K$.

---

### 6. Final Interpretation

In the infinite-width limit:

- The network remains nonlinear in input space.
- But it becomes linear in parameter space.
- Its training dynamics are governed by a fixed kernel.
- Learning corresponds to kernel regression in the gradient feature space.

Therefore, infinite-width neural networks interpolate between two perspectives:

- As nonlinear function approximators in input space.
- As linear models in an infinite-dimensional feature space determined at initialization.

The NTK formalism makes this bridge precise.

---

### 7. Example

To make the Neural Tangent Kernel perspective concrete, we now examine a one-hidden-layer ReLU network and study the rank of its empirical NTK matrix as the width varies.

The model has the form

$$
f(x) = \frac{1}{\sqrt{m}} \sum_{j=1}^{m} w_{2,j} \, \text{ReLU}(x w_{1,j} + b_{1,j}),
$$

where $m$ is the width of the hidden layer. As $m$ increases the number of parameters and the dimensionality of the gradient feature space grows.

---

#### Why Study the Rank of the NTK?

In the NTK regime, training dynamics are governed by the kernel

$$
K(x_i, x_j)
=
\nabla_\theta f(x_i, \theta_0)^T
\nabla_\theta f(x_j, \theta_0),
$$

which can be written in matrix form as

$$
K = \Phi \Phi^T,
$$

where

$$
\Phi_{i,:} = \nabla_\theta f(x_i, \theta_0).
$$

The rank of $K$ is fundamental because

$$
\mathrm{rank}(K) = \mathrm{rank}(\Phi),
$$

meaning it measures how many **linearly independent gradient feature directions** exist across the dataset.

Studying the rank tells us:

- How many independent directions in function space the model can move along during training.
- Whether the gradient features span a rich space or collapse into a low-dimensional subspace.
- Whether the model can interpolate arbitrary labels (full rank case).
- How architecture and data geometry constrain expressivity in the NTK regime.

Even though the parameter dimension grows with width, the effective dimensionality of the learning dynamics may remain limited. By computing the empirical NTK matrix for different widths and measuring its rank, we directly observe how increasing width affects the independent directions available to gradient descent.

In [28]:

class NTK(nn.Module):
    def __init__(self, input_dim=1, width=20):
        super().__init__()
        
        self.width = width
        self.w1 = nn.Parameter(torch.randn(1, width))
        self.b1 = nn.Parameter(torch.randn(1, width))
        self.w2 = nn.Parameter(torch.randn(width, 1))        

    def forward(self, x):
        h1 = x @ self.w1 + self.b1
        sigma = torch.relu(h1)
        output = (sigma @ self.w2) / math.sqrt(self.width)
        return output
                 
def gradient_features(model, x):
    model.zero_grad()
    y = model(x)
    y = y.sum()
    y.backward()
    
    g1 = model.w1.grad.flatten()
    g2 = model.w2.grad.flatten()
    
    phi = torch.cat([g1, g2])
    return phi.detach()

def build_feature_matrix(model, X):
    features = [gradient_features(model, X[i : i+1]) for i in range(len(X))]    
    return torch.stack(features)

sample_size = 200
x = torch.linspace(start=-torch.pi, end=torch.pi, steps=sample_size).reshape(-1, 1)
y = torch.sin(torch.pi * x)

print('Small m -> low kernel rank')
for i in range(10, 110, 10):
    model = NTK(width=i)
    phi = build_feature_matrix(model, x)
    k = phi @ phi.T
    
    print(f'\tFor {i} neurons we have a rank of {torch.linalg.matrix_rank(k)}')

print('\nHigh m -> high kernel rank')
for i in range(100, 2100, 100):
    model = NTK(width=i)
    phi = build_feature_matrix(model, x)
    k = phi @ phi.T
    
    print(f'\tFor {i} neurons we have a rank of {torch.linalg.matrix_rank(k)}')



Small m -> low kernel rank
	For 10 neurons we have a rank of 12
	For 20 neurons we have a rank of 15
	For 30 neurons we have a rank of 16
	For 40 neurons we have a rank of 23
	For 50 neurons we have a rank of 23
	For 60 neurons we have a rank of 20
	For 70 neurons we have a rank of 22
	For 80 neurons we have a rank of 28
	For 90 neurons we have a rank of 27
	For 100 neurons we have a rank of 32

High m -> high kernel rank
	For 100 neurons we have a rank of 31
	For 200 neurons we have a rank of 32
	For 300 neurons we have a rank of 39
	For 400 neurons we have a rank of 43
	For 500 neurons we have a rank of 42
	For 600 neurons we have a rank of 46
	For 700 neurons we have a rank of 48
	For 800 neurons we have a rank of 51
	For 900 neurons we have a rank of 51
	For 1000 neurons we have a rank of 49
	For 1100 neurons we have a rank of 45
	For 1200 neurons we have a rank of 48
	For 1300 neurons we have a rank of 50
	For 1400 neurons we have a rank of 48
	For 1500 neurons we have a rank of 5

This experiment illustrates how network width directly influences the effective dimensionality of learning in the NTK regime.

For small widths $m$, the empirical NTK matrix has relatively low rank. This indicates that the gradient feature vectors span only a limited number of independent directions. In this regime, the model behaves like kernel regression in a low-dimensional feature space, meaning gradient descent can only move along a restricted set of function directions.

As width increases, the kernel rank generally increases. This reflects the growth of the gradient feature space: wider networks produce more diverse gradient directions at initialization. Consequently, the space of functions accessible through linearized training becomes richer.

However, several important observations emerge:

- The rank does not grow linearly with width.
- It fluctuates due to randomness in initialization and numerical effects.
- It eventually stabilizes and grows slowly relative to $m$.

This behavior confirms a key NTK insight:

Even though the parameter dimension grows with width, the effective dimensionality of learning is determined by how many independent gradient feature directions the data induces.

In the infinite-width limit, the network does not gain unlimited expressive power in the NTK regime. Instead, it converges to kernel regression with a fixed kernel, whose complexity is constrained by both the architecture and the geometry of the data.

Thus, increasing width enriches the available feature space, but the rank of the NTK reveals the true functional degrees of freedom available to gradient descent.